# V5: Probabilistic Forecasting (Safety Stock Prediction)

## The Business Problem
Standard ML models generate **point forecasts** (e.g., \"We will sell 50 units\"). 
However, supply chain managers care about **uncertainty and risk**. If we stock exactly 50 units and demand spikes to 60, we face a stockout and lose revenue. 

To solve this, we must upgrade our models to output **Probabilistic Forecasts**. We need prediction intervals: e.g., \"We expect to sell 50 units, but there is a 10% chance we sell 80 units.\" 
The 90th percentile forecast (P90) becomes the **Safety Stock**.

## The Technical Solution: Quantile Regression
Instead of transitioning to a massive deep learning architecture (like Temporal Fusion Transformers), we can upgrade our Kaggle-winning `LightGBM` model to predict multiple quantiles simultaneously.

We will train three separate LightGBM models using the `quantile` objective:
- `alpha = 0.50` (Median: The baseline forecast)
- `alpha = 0.10` (P10: The pessimistic/low-demand scenario)
- `alpha = 0.90` (P90: The upper bound / Safety Stock scenario)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

## 1. Mock Data Generation (For Local/Standalone Testing)
In a real Kaggle environment, you would load the massive M5 dataset here and engineer the lag features. For demonstration of the probabilistic technique, we will generate a synthetic volatile time series mimicking the intermittent demand of a niche electronic item.

In [ ]:
np.random.seed(42)
days = 1000
date_rng = pd.date_range(start='1/1/2020', periods=days, freq='D')

# Create base demand with some seasonality and noise
base_demand = 50 + 20 * np.sin(np.arange(days) * (2 * np.pi / 365.25)) 
noise = np.random.normal(0, 15, days)
spikes = np.where(np.random.rand(days) > 0.95, np.random.randint(40, 100, days), 0)

demand = np.maximum(base_demand + noise + spikes, 0).astype(int)

df = pd.DataFrame({'date': date_rng, 'sales': demand})
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

# Feature Engineering: Lags & Rolling Means
df['lag_1'] = df['sales'].shift(1)
df['lag_7'] = df['sales'].shift(7)
df['rolling_mean_7'] = df['sales'].shift(1).rolling(window=7).mean()
df['rolling_std_7'] = df['sales'].shift(1).rolling(window=7).std()

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.head())

## 2. Train-Test Split
We will hold out the last 60 days to evaluate the accuracy of our prediction intervals.

In [ ]:
train_size = len(df) - 60
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

features = ['day_of_week', 'month', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7']
target = 'sales'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

## 3. Training the Quantile LightGBM Models
We train three distinct models, changing only the `alpha` parameter in the `quantile` objective.

In [ ]:
def train_quantile_model(X, y, alpha):
    model = lgb.LGBMRegressor(
        objective='quantile',
        alpha=alpha,
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )
    model.fit(X, y)
    return model

print("Training P10 (Pessimistic) Model...")
model_p10 = train_quantile_model(X_train, y_train, alpha=0.10)

print("Training P50 (Median) Model...")
model_p50 = train_quantile_model(X_train, y_train, alpha=0.50)

print("Training P90 (Safety Stock) Model...")
model_p90 = train_quantile_model(X_train, y_train, alpha=0.90)

print("All models trained successfully!")

## 4. Forecasting & Visualization
We predict the P10, P50, and P90 values for the test set and plot the resulting "Fan Chart".

In [ ]:
test_df = test_df.copy()
test_df['forecast_p10'] = model_p10.predict(X_test)
test_df['forecast_p50'] = model_p50.predict(X_test)
test_df['forecast_p90'] = model_p90.predict(X_test)

# Ensure logical ordering (sometimes tree models can cross predictions slightly)
test_df['forecast_p10'] = np.minimum(test_df['forecast_p10'], test_df['forecast_p50'])
test_df['forecast_p90'] = np.maximum(test_df['forecast_p90'], test_df['forecast_p50'])

plt.figure(figsize=(15, 6))
plt.plot(test_df['date'], test_df['sales'], label='Actual Sales (Ground Truth)', color='black', marker='o', alpha=0.7)
plt.plot(test_df['date'], test_df['forecast_p50'], label='Median Forecast (P50)', color='blue', linewidth=2)

plt.fill_between(
    test_df['date'], 
    test_df['forecast_p10'], 
    test_df['forecast_p90'], 
    color='blue', 
    alpha=0.2, 
    label='80% Confidence Interval (P10 to P90)'
)

plt.title('Probabilistic Demand Forecast via Quantile LightGBM', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 5. Business Interpretation: Safety Stock

If we only generated a point forecast (the blue line), we would stock the shelves exactly to the median expectation. On days where actual sales spiked (the black dots that jump outside the shaded region), we would run out of inventory.

By predicting the **P90** (the upper boundary of the shaded region), we give the supply chain a dynamic **Safety Stock target**. 
- On volatile days where the model detects high uncertainty, the P90 line naturally rises, instructing the warehouse to stock more buffer inventory.
- On stable days, the P90 line tightly hugs the median, saving the company money by preventing overstocking.

This transforms our ML project from an academic accuracy exercise into a highly actionable Operations Research tool.